# Phase 4: LLMs & Prompt Engineering
## Day 19: StructuredOutputExtraction

Date: 2026-04-24

### Learning objectives
- Ask an LLM for JSON-only output.
- Parse JSON safely in Python.
- Validate extracted fields with Pydantic.
- Detect broken or incomplete outputs.
- Build simple retry and repair patterns.

In [ ]:
import json
import re
import textwrap
from typing import Optional, Any, Dict, List
from pprint import pprint

try:
    from pydantic import BaseModel, Field, ValidationError
    PYDANTIC_AVAILABLE = True
except Exception as error:
    BaseModel = object
    Field = None
    ValidationError = Exception
    PYDANTIC_AVAILABLE = False
    print("Pydantic is not installed. The notebook will use fallback validation.")

def show(title, content):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)
    print(textwrap.dedent(str(content)).strip())

def model_to_dict(model):
    if hasattr(model, "model_dump"):
        return model.model_dump()
    if hasattr(model, "dict"):
        return model.dict()
    return model

print("Setup complete.")
print("Pydantic available:", PYDANTIC_AVAILABLE)

In [ ]:
raw_campaign_summaries = [
    '''
    Campaign: Spring Coffee Push
    Channel: Instagram
    Spend: 1200 EUR
    Clicks: 3420
    Conversions: 184
    Notes: Strong CTR. Conversions improved after adding a limited-time discount.
    ''',
    '''
    Campaign Bank App Onboarding ran on Email.
    It spent 800 EUR, received 980 clicks, and generated 42 conversions.
    The subject line may be too generic.
    ''',
    '''
    Yoga Studio Trial used TikTok with 650 EUR spend.
    It produced 2100 clicks and 165 conversions.
    Short beginner-friendly videos performed well.
    '''
]

expected_schema = {
    "campaign": "string",
    "channel": "string or null",
    "spend_eur": "number or null",
    "clicks": "integer or null",
    "conversions": "integer or null",
    "notes": "string or null"
}

show("Sample raw text", raw_campaign_summaries[0])
print("\nExpected schema:")
pprint(expected_schema)

## 1. Why structured output matters

LLMs are good at reading messy text. Applications need predictable data.

Structured extraction turns free text into fields that your code can validate, store, filter, and analyze.

In [ ]:
bad_output = '''
The Spring Coffee Push campaign did well on Instagram. It spent about 1200 EUR and got 3420 clicks.
Conversions were 184, which seems strong.
'''

good_output = {
    "campaign": "Spring Coffee Push",
    "channel": "Instagram",
    "spend_eur": 1200,
    "clicks": 3420,
    "conversions": 184,
    "notes": "Strong CTR. Conversions improved after adding a limited-time discount."
}

show("Hard to parse prose", bad_output)
print("\nEasy to parse JSON-like dict:")
pprint(good_output)

In [ ]:
def conversion_rate(record):
    if not record.get("clicks") or not record.get("conversions"):
        return None
    return record["conversions"] / record["clicks"]

print("Conversion rate:", round(conversion_rate(good_output), 4))

## 2. Prompting for JSON-only output

A good extraction prompt includes task, text, schema, and strict output rules.

The most important rule is simple: return valid JSON only.

In [ ]:
def build_json_extraction_prompt(text, schema):
    return f'''
Task:
Extract campaign information from the text.

Text:
{text.strip()}

Rules:
- Return valid JSON only.
- Do not include markdown.
- Do not include explanations.
- Use null when a field is missing.
- Use numbers for numeric fields.

JSON schema:
{json.dumps(schema, indent=2)}
'''.strip()

prompt = build_json_extraction_prompt(raw_campaign_summaries[0], expected_schema)
show("JSON extraction prompt", prompt)

In [ ]:
mock_llm_json = '''
{
  "campaign": "Spring Coffee Push",
  "channel": "Instagram",
  "spend_eur": 1200,
  "clicks": 3420,
  "conversions": 184,
  "notes": "Strong CTR. Conversions improved after adding a limited-time discount."
}
'''

parsed = json.loads(mock_llm_json)
print(type(parsed))
pprint(parsed)

## 3. Safe JSON parsing

LLM output can be invalid. Your parser should fail clearly.

Use `try/except` so your pipeline can retry, repair, or log the problem.

In [ ]:
def safe_json_loads(text):
    try:
        return json.loads(text), None
    except json.JSONDecodeError as error:
        return None, str(error)

valid_text = '{"campaign": "Demo", "clicks": 100}'
broken_text = '{"campaign": "Demo", "clicks": 100,}'

for item in [valid_text, broken_text]:
    data, error = safe_json_loads(item)
    print("\nInput:", item)
    print("Data:", data)
    print("Error:", error)

In [ ]:
def extract_json_block(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    return match.group(0)

messy_output = '''
Here is the JSON:

```json
{
  "campaign": "Bank App Onboarding",
  "channel": "Email",
  "spend_eur": 800,
  "clicks": 980,
  "conversions": 42,
  "notes": "Subject line may be too generic."
}
```
'''

json_block = extract_json_block(messy_output)
print(json_block)
data, error = safe_json_loads(json_block)
print("\nParsed:")
pprint(data)
print("Error:", error)

## 4. Pydantic validation

Parsing only checks if the text is valid JSON. Validation checks if the data has the right fields and types.

Pydantic is useful because it creates a clear contract between the LLM and your code.

In [ ]:
if PYDANTIC_AVAILABLE:
    class CampaignExtraction(BaseModel):
        campaign: str
        channel: Optional[str] = None
        spend_eur: Optional[float] = Field(default=None, ge=0)
        clicks: Optional[int] = Field(default=None, ge=0)
        conversions: Optional[int] = Field(default=None, ge=0)
        notes: Optional[str] = None
else:
    CampaignExtraction = None

def fallback_validate_campaign(data):
    required = ["campaign", "channel", "spend_eur", "clicks", "conversions", "notes"]
    errors = []
    for key in required:
        if key not in data:
            errors.append(f"Missing field: {key}")
    for key in ["spend_eur", "clicks", "conversions"]:
        value = data.get(key)
        if value is not None and not isinstance(value, (int, float)):
            errors.append(f"{key} must be numeric or null")
        if isinstance(value, (int, float)) and value < 0:
            errors.append(f"{key} must be non-negative")
    if not isinstance(data.get("campaign"), str):
        errors.append("campaign must be a string")
    if errors:
        raise ValueError(errors)
    return data

def validate_campaign(data):
    if PYDANTIC_AVAILABLE:
        return CampaignExtraction(**data)
    return fallback_validate_campaign(data)

valid_record = {
    "campaign": "Spring Coffee Push",
    "channel": "Instagram",
    "spend_eur": 1200,
    "clicks": 3420,
    "conversions": 184,
    "notes": "Strong CTR."
}

validated = validate_campaign(valid_record)
print("Validated record:")
pprint(model_to_dict(validated))

In [ ]:
invalid_record = {
    "campaign": "Broken Campaign",
    "channel": "Email",
    "spend_eur": -50,
    "clicks": "many",
    "conversions": 10,
    "notes": None
}

try:
    validate_campaign(invalid_record)
except Exception as error:
    print("Validation failed.")
    print(error)

## 5. Rule-based extraction as a local mock

In real projects, an LLM does the extraction.

For this notebook, we use a small rule-based extractor so every example runs without API keys.

In [ ]:
def find_number(pattern, text):
    match = re.search(pattern, text, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

def mock_extract_campaign(text):
    text_clean = " ".join(text.split())

    campaign = None
    patterns = [
        r"Campaign:\s*([A-Za-z\s]+?)\s+Channel:",
        r"Campaign\s+([A-Za-z\s]+?)\s+ran",
        r"([A-Za-z\s]+?)\s+used\s+TikTok"
    ]
    for pattern in patterns:
        match = re.search(pattern, text_clean, flags=re.IGNORECASE)
        if match:
            campaign = match.group(1).strip()
            break

    channel = None
    for candidate in ["Instagram", "Email", "TikTok", "Google", "LinkedIn"]:
        if candidate.lower() in text_clean.lower():
            channel = candidate
            break

    spend = find_number(r"(\d+)\s*EUR", text_clean)
    clicks = find_number(r"(\d+)\s+clicks", text_clean)
    conversions = find_number(r"(\d+)\s+conversions", text_clean)

    notes = None
    notes_match = re.search(r"Notes:\s*(.+)", text, flags=re.IGNORECASE | re.DOTALL)
    if notes_match:
        notes = " ".join(notes_match.group(1).split())
    elif "subject line" in text_clean.lower():
        notes = "The subject line may be too generic."
    elif "beginner-friendly" in text_clean.lower():
        notes = "Short beginner-friendly videos performed well."

    return {
        "campaign": campaign,
        "channel": channel,
        "spend_eur": spend,
        "clicks": clicks,
        "conversions": conversions,
        "notes": notes
    }

for text in raw_campaign_summaries:
    extracted = mock_extract_campaign(text)
    pprint(extracted)
    print()

In [ ]:
validated_records = []

for text in raw_campaign_summaries:
    extracted = mock_extract_campaign(text)
    validated = validate_campaign(extracted)
    validated_records.append(model_to_dict(validated))

pprint(validated_records)

## 6. Retry pattern

A retry pattern sends a better prompt after parsing or validation fails.

The retry should include the error and repeat the schema.

In [ ]:
def make_retry_prompt(original_text, bad_output, error_message, schema):
    return f'''
Your previous output could not be parsed or validated.

Original text:
{original_text.strip()}

Previous output:
{bad_output}

Error:
{error_message}

Fix the output.

Rules:
- Return valid JSON only.
- Do not include markdown.
- Do not include explanations.
- Use this exact schema:
{json.dumps(schema, indent=2)}
'''.strip()

bad_output = '{"campaign": "Demo", "clicks": "many",}'
data, parse_error = safe_json_loads(bad_output)

retry_prompt = make_retry_prompt(
    original_text=raw_campaign_summaries[1],
    bad_output=bad_output,
    error_message=parse_error,
    schema=expected_schema
)

show("Retry prompt", retry_prompt)

In [ ]:
def mock_llm_retry_response(original_text):
    # In a real app, this would call the LLM again.
    return json.dumps(mock_extract_campaign(original_text), indent=2)

retry_output = mock_llm_retry_response(raw_campaign_summaries[1])
print(retry_output)

data, error = safe_json_loads(retry_output)
validated = validate_campaign(data)

print("\nValidated retry result:")
pprint(model_to_dict(validated))

## 7. Repair pattern

A repair pattern tries to fix small formatting issues before retrying the LLM.

This is useful for cheap errors, such as trailing commas or markdown fences.

In [ ]:
def repair_json_text(text):
    repaired = text.strip()
    repaired = repaired.replace("```json", "").replace("```", "").strip()
    repaired = re.sub(r",\s*}", "}", repaired)
    repaired = re.sub(r",\s*]", "]", repaired)
    return repaired

broken_outputs = [
    '{"campaign": "Demo", "clicks": 100,}',
    '''```json
    {"campaign": "Demo", "clicks": 100}
    ```'''
]

for broken in broken_outputs:
    repaired = repair_json_text(broken)
    data, error = safe_json_loads(repaired)
    print("\nOriginal:", broken)
    print("Repaired:", repaired)
    print("Parsed:", data)
    print("Error:", error)

In [ ]:
def parse_validate_or_repair(text):
    data, error = safe_json_loads(text)

    if error:
        repaired = repair_json_text(text)
        data, error = safe_json_loads(repaired)

    if error:
        return None, f"Parse error: {error}"

    try:
        validated = validate_campaign(data)
        return model_to_dict(validated), None
    except Exception as validation_error:
        return None, f"Validation error: {validation_error}"

almost_good = '''
```json
{
  "campaign": "Spring Coffee Push",
  "channel": "Instagram",
  "spend_eur": 1200,
  "clicks": 3420,
  "conversions": 184,
  "notes": "Strong CTR.",
}
```
'''

result, error = parse_validate_or_repair(almost_good)
pprint(result)
print("Error:", error)

## 8. Mini pipeline

Now combine the pieces: prompt, extract, parse, validate, repair, and collect results.

This is the basic shape of an information extraction pipeline.

In [ ]:
def extraction_pipeline(text):
    prompt = build_json_extraction_prompt(text, expected_schema)

    # Mock LLM output. Replace this with a real LLM call in production.
    llm_output = json.dumps(mock_extract_campaign(text), indent=2)

    data, error = parse_validate_or_repair(llm_output)

    return {
        "prompt_preview": prompt[:160] + "...",
        "raw_output": llm_output,
        "data": data,
        "error": error
    }

pipeline_results = [extraction_pipeline(text) for text in raw_campaign_summaries]

for item in pipeline_results:
    print("\nPrompt preview:", item["prompt_preview"])
    print("Error:", item["error"])
    print("Data:")
    pprint(item["data"])

In [ ]:
import pandas as pd

records = [item["data"] for item in pipeline_results if item["error"] is None]
df = pd.DataFrame(records)
df["conversion_rate"] = df["conversions"] / df["clicks"]

df

## Tricky bits

Structured output fails in predictable ways.

Common issues are markdown fences, trailing commas, missing fields, wrong types, and prose mixed with JSON.

In [ ]:
tricky_outputs = {
    "markdown_fence": '''```json\n{"campaign": "Demo", "clicks": 100}\n```''',
    "trailing_comma": '{"campaign": "Demo", "clicks": 100,}',
    "prose_plus_json": 'Sure, here it is: {"campaign": "Demo", "clicks": 100}',
    "wrong_type": '{"campaign": "Demo", "clicks": "one hundred"}'
}

for name, output in tricky_outputs.items():
    print("\nCase:", name)
    json_block = extract_json_block(output) or output
    repaired = repair_json_text(json_block)
    data, error = safe_json_loads(repaired)
    print("Parsed:", data)
    print("Parse error:", error)

In [ ]:
minimum_required_fields = ["campaign", "channel", "spend_eur", "clicks", "conversions", "notes"]

def check_missing_fields(data, required_fields):
    return [field for field in required_fields if field not in data]

partial_output = {"campaign": "Demo", "clicks": 100}
missing = check_missing_fields(partial_output, minimum_required_fields)

print("Missing fields:", missing)

if missing:
    print("Retry needed. The model did not follow the schema.")

## Trick questions

1. Is valid JSON always valid structured data?

<details>
<summary>Answer</summary>

No. JSON can parse correctly but still have missing fields, wrong types, or impossible values.

</details>

2. Why should extraction prompts say "JSON only"?

<details>
<summary>Answer</summary>

Because extra prose or markdown can break parsers and downstream pipelines.

</details>

3. Should every broken output be repaired locally?

<details>
<summary>Answer</summary>

No. Repair small formatting issues locally. Retry the LLM for missing fields, unclear values, or wrong semantics.

</details>

4. What does Pydantic add after `json.loads()`?

<details>
<summary>Answer</summary>

It validates the schema, field types, required fields, and optional constraints.

</details>

5. Why include the error message in a retry prompt?

<details>
<summary>Answer</summary>

It tells the model exactly what to fix, which makes the retry more targeted.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Create a JSON-only instruction.

json_rule = ___

assert isinstance(json_rule, str)
assert "json" in json_rule.lower()
assert "only" in json_rule.lower()
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Parse a JSON string into a Python dict.

text = '{"campaign": "Demo Campaign", "clicks": 250}'
data = ___

assert isinstance(data, dict)
assert data["campaign"] == "Demo Campaign"
assert data["clicks"] == 250
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Extract the JSON block from messy output.

messy = 'Here is the result: {"campaign": "Demo", "clicks": 100}'
json_part = ___

assert json_part == '{"campaign": "Demo", "clicks": 100}'
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Complete a campaign record with all required fields.

record = {
    "campaign": "Demo",
    "channel": ___,
    "spend_eur": ___,
    "clicks": ___,
    "conversions": ___,
    "notes": ___
}

validated = validate_campaign(record)

assert model_to_dict(validated)["campaign"] == "Demo"
assert model_to_dict(validated)["clicks"] >= 0
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Repair a JSON string with a trailing comma.

broken = '{"campaign": "Demo", "clicks": 100,}'
repaired = ___
parsed = json.loads(repaired)

assert parsed["clicks"] == 100
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Build a retry prompt.

retry = make_retry_prompt(
    original_text=raw_campaign_summaries[0],
    bad_output='{"campaign": "Demo",}',
    error_message=___,
    schema=expected_schema
)

assert "Error:" in retry
assert "Return valid JSON only" in retry
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Calculate conversion rate from a validated record.

record = {
    "campaign": "Demo",
    "channel": "Email",
    "spend_eur": 500,
    "clicks": 1000,
    "conversions": 75,
    "notes": None
}

rate = ___

assert abs(rate - 0.075) < 1e-9
print("Exercise 7 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
json_rule = "Return valid JSON only."
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
data = json.loads(text)
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
json_part = extract_json_block(messy)
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
record = {
    "campaign": "Demo",
    "channel": "Email",
    "spend_eur": 500,
    "clicks": 1000,
    "conversions": 75,
    "notes": None
}
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
repaired = repair_json_text(broken)
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
error_message = "JSONDecodeError: trailing comma"
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
rate = record["conversions"] / record["clicks"]
```

</details>

## Cumulative review exercises

These mix topics from Days 9 to 18. Fill in `___` and run each cell.

In [ ]:
# Review 1: Classification metrics
# Calculate recall.

tp = 45
fn = 15

recall = ___

assert abs(recall - 0.75) < 1e-9
print("Review 1 passed.")

In [ ]:
# Review 2: SHAP
# Complete the idea.

shap_meaning = ___

assert "feature" in shap_meaning.lower()
assert "prediction" in shap_meaning.lower()
print("Review 2 passed.")

In [ ]:
# Review 3: TF-IDF
# Choose the vectorizer name.

vectorizer_name = ___

assert vectorizer_name == "TfidfVectorizer"
print("Review 3 passed.")

In [ ]:
# Review 4: Embeddings
# Select the similarity metric often used with embeddings.

similarity_metric = ___

assert similarity_metric.lower() == "cosine similarity"
print("Review 4 passed.")

In [ ]:
# Review 5: Hugging Face
# Fill the two common classes for model loading.

hf_classes = ___

assert "AutoTokenizer" in hf_classes
assert "AutoModel" in hf_classes
print("Review 5 passed.")

In [ ]:
# Review 6: Fine-tuning BERT
# Choose the common evaluation function from sklearn.

eval_function = ___

assert eval_function == "classification_report"
print("Review 6 passed.")

In [ ]:
# Review 7: Complaint classification
# Create labels for a simple support classifier.

labels = ___

assert isinstance(labels, list)
assert len(labels) >= 3
assert all(isinstance(label, str) for label in labels)
print("Review 7 passed.")

In [ ]:
# Review 8: OpenAI API
# Fill the role that usually sets behavior and rules.

behavior_role = ___

assert behavior_role == "system"
print("Review 8 passed.")

In [ ]:
# Review 9: Ollama
# Fill the default local Ollama base URL.

ollama_base_url = ___

assert ollama_base_url == "http://localhost:11434"
print("Review 9 passed.")

In [ ]:
# Review 10: Prompt engineering
# Pick the prompting style that uses examples.

prompting_style = ___

assert prompting_style.lower() == "few-shot"
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
recall = tp / (tp + fn)

# Review 2
shap_meaning = "SHAP explains how each feature contributes to a prediction."

# Review 3
vectorizer_name = "TfidfVectorizer"

# Review 4
similarity_metric = "cosine similarity"

# Review 5
hf_classes = ["AutoTokenizer", "AutoModel"]

# Review 6
eval_function = "classification_report"

# Review 7
labels = ["billing", "technical", "delivery"]

# Review 8
behavior_role = "system"

# Review 9
ollama_base_url = "http://localhost:11434"

# Review 10
prompting_style = "few-shot"
```

</details>

In [ ]:
cheat_sheet = '''
DAY 19 CHEAT SHEET: STRUCTURED OUTPUT EXTRACTION

Prompt rules:
- Give the task.
- Provide the raw text.
- Provide the JSON schema.
- Say: Return valid JSON only.
- Say: Use null for missing values.

Parsing:
- json.loads(text) converts JSON text to Python objects.
- Use try/except for JSONDecodeError.
- Extract JSON blocks when the model adds extra text.

Validation:
- Parsing checks JSON syntax.
- Validation checks fields, types, and constraints.
- Pydantic is a good validation tool.

Repair:
- Remove markdown fences.
- Remove trailing commas.
- Repair only cheap formatting issues.

Retry:
- Retry when fields are missing, types are wrong, or values are unclear.
- Include the original text, bad output, error, and schema.
'''

print(cheat_sheet)

## Next up: Day 20 — InformationExtractionProject

You will build a campaign summary to structured JSON pipeline as a mini project.